# 🫀 퀘스트 46 · Q7 — **SVDB 가 S 채점 코호트가 될 수 있나**

| | **MedKOS / `notebooks/quest46_q7_svdb_cohort.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | `ailab-2026-0047` (Q1 — DS2·INCART 둘 다 기각) |
| 규약 | `SCORING_RULES.md` **R11 · R11-b · R12 · R13** |

## 왜 이 실험인가

Q1 이 두 코호트를 기각했다:

| 코호트 | 왜 안 되나 |
|---|---|
| MIT-BIH DS2 | **구조적 상한 13명** — S>0 인 레코드 16/22 · 중앙값 4.5 |
| INCART | 20명(GMIN≤5)과 최대 SE 0.10(GMIN≥10)이 **교집합 없음**. 양성 3~6개짜리 레코드가 SE 를 0.27·0.16 으로 밀어올린다 |

남은 후보가 **SVDB** 다 — 이름부터 supraventricular 인 전용 DB(78레코드).

## 무엇을 하나 · 무엇을 못 하나

**한다**: 주석(`.atr`)만 내려받아 레코드별 S 개수를 세고 **R13 분포**를 낸다.
신호(`.dat`)는 안 받는다. 학습 0회.

**못 한다**: SE 는 예측이 있어야 잰다. 그래서 Q1 실측을 **필요조건**으로 쓴다 —
부트 최대 SE 가 양성 3개에서 **0.2661** · 6개에서 **0.1577** 이었으므로,
양성이 한 자리인 개체가 섞이면 상한 0.10 을 못 지킨다.
→ `GMIN_SCREEN = 10` 에서 채점 개체가 20개 이상인지를 먼저 본다.

**여기서 떨어지면 SE 를 볼 것도 없다. 통과해도 Q7-B 에서 실제 SE 를 재야 확정이다.**

## 사전등록

| 관문 | 내용 |
|---|---|
| **Q7-1** | S ≥ 10 인 레코드가 **20개 이상** |
| **Q7-2** | 지배 지분 ≤ 50% (R11-3) |
| **Q7-3** | GMIN=10 채점 개체가 덮는 S ≥ 70% |
| **Q7-4** | SVDB 가 DS2·INCART 보다 채점 개체를 많이 준다 |

⚠️ AAMI 매핑은 `svdb_prep.py:29` 와 **동일**하게 쓴다 — 다르면 DS2·INCART 와 비교가
안 된다. F·Q·페이스박은 그 매핑에서 제외된다(`svdb_labels.py` 가 지적한 한계).

⚠️ SVDB 는 **128Hz**, MIT-BIH 는 360Hz 다(`svdb_prep.py` F4). 분포 비교에는 무관하지만
모델을 태울 때는 대역 차이를 반드시 병기한다.

✅ SVDB 는 **레코드 = 환자 1:1** 이다(F3) — INCART 의 L3(75레코드 = 32환자) 문제가 없다.


In [ ]:
# CELL 0 — 공용 사전점검 (pipelines/SCORING_RULES.md)
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    """사전등록 관문의 유일한 계약: 지지 / 기각 / **미결**."""
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def t_ci(v, conf=0.95):
    v = np.asarray([x for x in v if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2: return m, np.nan, np.nan
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * v.std(ddof=1) / np.sqrt(n))
    return m, m - h, m + h

class AssetError(RuntimeError): pass
print("CELL 0 ✅ decide · t_ci 준비")

In [ ]:
# CELL 1 — 【Q7】 SVDB 의 S 분포 — **필요조건 스크리닝** (학습 0회 · 신호 다운로드 없음)
#
#  왜: Q1 이 DS2·INCART 를 둘 다 기각했다. 남은 후보는 SVDB(상심실성 부정맥 **전용**
#      DB, 78레코드)다. 여기서도 S 가 소수 레코드에 몰려 있으면 같은 벽에 부딪힌다.
#
#  ★ 이 셀은 **주석(.atr)만** 내려받는다. 신호(.dat)는 안 받는다 — 라벨만 있으면
#    분포를 잴 수 있고, 78개 주석 파일은 몇 MB 다.
#
#  ★ 무엇을 **못** 하나: SE 는 예측이 있어야 잰다. 그래서 여기서는 Q1 실측을
#    **필요조건**으로 쓴다 — 부트 최대 SE 가 양성 3개에서 0.2661 · 6개에서 0.1577
#    이었으므로, 양성이 한 자리인 개체가 섞이면 상한 0.10 을 넘길 위험이 크다.
#    → `GMIN_SCREEN = 10` 에서 채점 개체가 20개 이상인지를 먼저 본다.
#      통과해야 Q7-B(예측 생성 + SE 실측)로 간다. 통과 못 하면 SVDB 도 기각이다.
import os, sys, json, subprocess, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun
try:
    import wfdb
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb
import numpy as np

# ★ svdb_prep.py:29 와 **똑같은** 매핑. 다르면 DS2·INCART 와 비교가 안 된다.
_AAMI = {'N':0,'L':0,'R':0,'e':0,'j':0, 'A':1,'a':1,'J':1,'S':1, 'V':2,'E':2}
GMIN_SCREEN = 10       # Q1 실측 기반 필요조건 — 양성 한 자리는 SE 가 못 버틴다
N_MIN       = 20
GMINS       = [2, 5, 10, 15, 20, 30, 50, 75, 100, 200, 500]

CONFIG = dict(
    exp="quest46_q7_svdb_cohort", quest="ailab-2026-0046", step="svdb-cohort-check",
    parent_exp=["quest46_q1_gmin_resolution", "ailab-2026-0047"],
    purpose=("Q1 이 DS2(구조적 상한 13명)와 INCART(20명과 최대SE 0.10 이 교집합 없음)를 "
             "둘 다 기각했다. SVDB 가 S 채점 코호트가 될 수 있는지 **분포로 먼저** 본다"),
    dataset="MIT-BIH Supraventricular Arrhythmia Database (svdb) · 78레코드 · 주석만",
    aami_map=_AAMI, gmin_screen=GMIN_SCREEN, n_min=N_MIN,
    metric_rule="R13 — 지배지분·양성>0 개체 수·중앙값·상위k 커버 지점을 함께 낸다",
    predictions={
        "Q7-1": f"S 양성 ≥ {GMIN_SCREEN} 인 레코드가 **{N_MIN}개 이상**",
        "Q7-2": "지배 지분 ≤ 50% (R11-3)",
        "Q7-3": f"GMIN={GMIN_SCREEN} 채점 개체가 덮는 S ≥ 70%",
        "Q7-4": "SVDB 가 DS2·INCART 보다 채점 개체를 많이 준다"},
    caveat=("SE 는 예측이 있어야 잰다 — 여기서는 **필요조건**만 본다. 통과해도 Q7-B 에서 "
            "실제 SE 를 재야 확정이다. 반대로 여기서 떨어지면 SE 를 볼 것도 없다. "
            "SVDB 는 128Hz 이고 MIT-BIH 는 360Hz 다(svdb_prep.py F4) — 분포 비교에는 "
            "무관하지만 나중에 모델을 태울 때는 대역 차이를 반드시 병기한다"))
run = MedKOSRun("quest46_q7_svdb", CONFIG, project=PROJECT)
run.log(f"설정 ✅ GMIN_SCREEN {GMIN_SCREEN} · N_MIN {N_MIN}")

In [ ]:
# CELL 2 — 【Q7】 SVDB 주석 전수 조사
CACHE_J = os.path.join(PROJECT, "data", "svdb_ann_counts.json")
if os.path.exists(CACHE_J):
    CNT = {int(k): {int(kk): vv for kk, vv in v.items()}
           for k, v in json.load(open(CACHE_J)).items()}
    run.log(f"주석 카운트 캐시 적중 — {len(CNT)}레코드 ({CACHE_J})")
else:
    recs = wfdb.get_record_list("svdb")
    run.log(f"SVDB 레코드 {len(recs)}개 — 주석만 내려받는다(.dat 안 받음)")
    CNT, bad = {}, []
    for i, r in enumerate(recs):
        try:
            a = wfdb.rdann(r, "atr", pn_dir="svdb")
        except Exception as e:
            bad.append((r, str(e)[:60])); continue
        c = {0: 0, 1: 0, 2: 0}
        for sym in a.symbol:
            k = _AAMI.get(sym)
            if k is not None:
                c[k] += 1
        CNT[int(r)] = c
        if (i + 1) % 20 == 0:
            run.log(f"  {i+1}/{len(recs)} …")
    if bad:
        run.log(f"  ⚠️ 실패 {len(bad)}건: {bad[:3]}")
    os.makedirs(os.path.dirname(CACHE_J), exist_ok=True)
    json.dump({str(k): v for k, v in CNT.items()}, open(CACHE_J, "w"))
    run.log(f"  캐시 저장 → {CACHE_J}")

TOT = {k: sum(c[k] for c in CNT.values()) for k in (0, 1, 2)}
run.log(f"\n  SVDB 총 {sum(TOT.values()):,}비트 · N {TOT[0]:,} · **S {TOT[1]:,}** · V {TOT[2]:,}")
run.log(f"  S 기저율 {TOT[1]/max(sum(TOT.values()),1):.4f}")
run.log(f"  ⚠️ F·Q·페이스박은 svdb_prep.py 매핑에서 **제외**된다(svdb_labels.py 지적)")

In [ ]:
# CELL 3 — 【Q7-A】 R13 분포 통계 + 세 코호트 대조
def dist_stats(counts):
    """R13: 지배지분 · 양성>0 개체 수 · 중앙값 · 상위 k 가 50%·90% 를 덮는 지점."""
    v = np.array(sorted(counts.values(), reverse=True), float); tot = v.sum()
    if tot <= 0:
        return dict(total=0, n_unit=len(counts), n_pos=0, dom=np.nan, med=0.0, k50=0, k90=0)
    cum = np.cumsum(v) / tot
    return dict(total=int(tot), n_unit=len(counts), n_pos=int((v > 0).sum()),
                dom=float(v[0] / tot), med=float(np.median(v)),
                k50=int(np.searchsorted(cum, .50) + 1),
                k90=int(np.searchsorted(cum, .90) + 1))

def gmin_table(counts, gmins):
    v = np.array(list(counts.values()), float); tot = v.sum()
    out = []
    for g in gmins:
        k = v[v >= g]
        out.append(dict(gmin=g, n=int(len(k)),
                        cov=float(k.sum() / tot) if tot else 0.0,
                        dom=float(k.max() / k.sum()) if len(k) and k.sum() else np.nan))
    return out

SVDB_S = {r: c[1] for r, c in CNT.items()}
# 실험22-A·Q1 실측(전수) — 대조군. 지어낸 값이 아니라 forensics/Q1 출력에서 옮긴 것이다
DS2_S = {100:33,103:2,105:0,111:0,113:6,117:1,121:1,123:0,200:30,202:55,210:22,
         212:0,213:28,214:0,219:7,221:0,222:209,228:3,231:1,232:1382,233:7,234:50}

run.log("\n" + "=" * 100)
run.log("【Q7-A】 R13 분포 — 지배지분만으로는 '고르다' 를 판단할 수 없다")
run.log("=" * 100)
run.log(f"  {'코호트':<18}{'양성':>8}{'개체':>6}{'양성>0':>8}{'지배지분':>9}"
        f"{'중앙값':>8}{'k50':>5}{'k90':>5}")
STAT = {}
for nm, cc in (("MIT-BIH DS2 · S", DS2_S), ("**SVDB · S**", SVDB_S)):
    s = dist_stats(cc); STAT[nm] = s
    run.log(f"  {nm:<18}{s['total']:>8,}{s['n_unit']:>6}{s['n_pos']:>8}"
            f"{s['dom']:>9.1%}{s['med']:>8.4g}{s['k50']:>5}{s['k90']:>5}")
run.log(f"  {'INCART · S (Q1)':<18}{1958:>8,}{75:>6}{36:>8}{0.301:>9.1%}"
        f"{0:>8}{2:>5}{8:>5}   ← ailab-2026-0047 실측")

run.log("\n  ── GMIN 별 채점 개체 (필요조건 스크리닝) ──")
run.log(f"  {'GMIN':>5}{'SVDB':>7}{'커버':>8}{'지배':>8}   {'INCART(Q1)':>11}{'DS2(Q1)':>9}")
Q1_INC = {2:27,5:20,10:12,15:12,20:11,30:10,50:7,75:6,100:6,200:2}
Q1_DS2 = {2:13,5:11,10:8,15:8,20:8,30:6,50:4,75:2,100:2,200:2}
SV = gmin_table(SVDB_S, GMINS)
for r in SV:
    g = r["gmin"]
    run.log(f"  {g:>5}{r['n']:>7}{r['cov']:>8.1%}{r['dom']:>8.1%}"
            f"   {Q1_INC.get(g,'—'):>11}{Q1_DS2.get(g,'—'):>9}")

In [ ]:
# CELL 4 — 【Q7 채점】 사전등록 관문
sel = next((r for r in SV if r["gmin"] == GMIN_SCREEN), None)
s_all = STAT["**SVDB · S**"]
run.log("\n" + "=" * 100)
run.log("【Q7 사전등록 채점】  ※ 필요조건만 본다 — 통과해도 Q7-B 에서 SE 를 재야 확정")
run.log("=" * 100)
VERD = {}
def g_(k, ok, d):
    VERD[k] = "✅ 지지" if ok else "❌ 기각"
    run.log(f"  {k:<7}{VERD[k]}  {d}")

g_("Q7-1", sel is not None and sel["n"] >= N_MIN,
   f"S ≥ {GMIN_SCREEN} 인 레코드 **{sel['n'] if sel else 0}개** ≥ {N_MIN}"
   f"  (INCART 12개 · DS2 8개)")
g_("Q7-2", np.isfinite(s_all["dom"]) and s_all["dom"] <= 0.50,
   f"지배 지분 **{s_all['dom']:.1%}** ≤ 50%")
g_("Q7-3", sel is not None and sel["cov"] >= 0.70,
   f"GMIN={GMIN_SCREEN} 채점 개체가 덮는 S **{sel['cov']:.1%}** ≥ 70%")
best_sv = max(r["n"] for r in SV)
g_("Q7-4", best_sv > max(max(Q1_INC.values()), max(Q1_DS2.values())),
   f"SVDB 최대 **{best_sv}개** vs INCART 27 · DS2 13")

run.log("\n  " + "  ".join(f"{k}: {v}" for k, v in VERD.items()))
ok_all = all(v.startswith("✅") for v in VERD.values())
if ok_all:
    run.log(f"\n  ✅ **필요조건 통과.** GMIN={GMIN_SCREEN} 에서 {sel['n']}개 확보."
            " → Q7-B(예측 생성 + SE 실측)로 간다.")
    run.log("     Q7-B 는 학습이 필요하다: MIT-BIH 학습 → SVDB zero-shot 예측(5시드).")
    run.log("     ⚠️ SVDB 128Hz vs MIT-BIH 360Hz — 대역 차이를 반드시 병기한다(svdb_prep F4).")
else:
    fail = [k for k, v in VERD.items() if v.startswith("❌")]
    run.log(f"\n  ❌ **{', '.join(fail)} 에서 걸렸다.** SVDB 도 S 채점 코호트가 못 된다.")
    run.log("     → Q8(가중 매크로)로 간다 — 개체를 버리는 대신 SE 역수로 가중해")
    run.log("       12명 코호트에서 정보를 최대한 짜내는 쪽이다.")

run.log("\n  ⚠️ 이건 **필요조건**이다. 양성 수만 봤고 SE 는 안 쟀다(예측이 없다).")
run.log("  ⚠️ SVDB 는 레코드=환자 1:1 이다(svdb_prep F3) — INCART 의 L3 문제가 없다.")
CONFIG["result"] = {"verdicts": VERD, "svdb_stats": s_all,
                    "svdb_gmin": SV, "screen_gmin": GMIN_SCREEN,
                    "per_record_S": {int(k): int(v) for k, v in sorted(SVDB_S.items())}}
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 그림 + 저장
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
v = np.array(sorted(SVDB_S.values(), reverse=True), float)
ax[0].bar(range(len(v)), v, color="#1f77b4")
ax[0].axhline(GMIN_SCREEN, ls="--", c="crimson", label=f"GMIN={GMIN_SCREEN}")
ax[0].set_yscale("symlog"); ax[0].set_xlabel("레코드 (S 많은 순)")
ax[0].set_ylabel("S 개수"); ax[0].set_title("① SVDB 레코드별 S — 몇 명에게 몰렸나")
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

for nm, tbl, st in (("SVDB", SV, "o-"),
                    ("INCART", [{"gmin": g, "n": n} for g, n in sorted(Q1_INC.items())], "s--"),
                    ("MIT-BIH DS2", [{"gmin": g, "n": n} for g, n in sorted(Q1_DS2.items())], "^:")):
    ax[1].plot([r["gmin"] for r in tbl], [r["n"] for r in tbl], st, label=nm)
ax[1].axhline(N_MIN, ls="--", c="crimson"); ax[1].axvline(GMIN_SCREEN, ls=":", c="gray")
ax[1].set_xscale("log"); ax[1].set_xlabel("GMIN_S"); ax[1].set_ylabel("채점 개체 수")
ax[1].set_title("② 코호트별 채점 개체 — 빨간 선을 넘어야 한다")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
fig.suptitle("Q7 — SVDB 가 S 채점 코호트가 될 수 있나 (필요조건)", y=1.02)
fig.tight_layout()
run.save_fig("q7_svdb_distribution", fig)
plt.show()
run.log("\n저장 완료 — " + run.dir)
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-cohort-check`")